In [4]:
import os
from Bio import SeqIO
from collections import defaultdict

def split_fasta_by_strain(input_fasta, output_folder):
    """
    Tách file FASTA lớn thành các file nhỏ theo từng chủng (Strain).
    """
    # Tạo thư mục đầu ra nếu chưa có
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)
        print(f"📂 Đã tạo thư mục: {output_folder}")

    # Gom nhóm các record theo Strain ID
    strain_groups = defaultdict(list)
    
    print(f"📖 Đang đọc file: {input_fasta}...")
    count = 0
    for record in SeqIO.parse(input_fasta, "fasta"):
        # Logic lấy Strain ID: phần trước dấu '|' hoặc '_'
        # Bạn có thể chỉnh lại r.id.split('|')[0] tùy theo format file của bạn
        sid = record.id.split('|')[0].split('_')[0]
        strain_groups[sid].append(record)
        count += 1

    print(f"✅ Đã đọc {count} sequences. Đang tiến hành ghi file...")

    # Ghi từng nhóm ra file FASTA riêng
    for sid, records in strain_groups.items():
        output_path = os.path.join(output_folder, f"{sid}.fasta")
        SeqIO.write(records, output_path, "fasta")
        print(f"   + [Saved] {sid}.fasta ({len(records)} contigs)")

    print(f"\n🚀 Hoàn tất! Đã tách thành {len(strain_groups)} chủng tại: {output_folder}")

if __name__ == "__main__":
    # CHỈNH ĐƯỜNG DẪN TẠI ĐÂY
    INPUT_FILE = "/Users/thanhnt/Desktop/Primer_tools/B_cereus_background/Bacillus_weihenstep.fasta"
    OUTPUT_DIR = "/Users/thanhnt/Desktop/Primer_tools/Non_sensulato/Bacillus_weihenstep"
    
    split_fasta_by_strain(INPUT_FILE, OUTPUT_DIR)

📂 Đã tạo thư mục: /Users/thanhnt/Desktop/Primer_tools/Non_sensulato/Bacillus_weihenstep
📖 Đang đọc file: /Users/thanhnt/Desktop/Primer_tools/B_cereus_background/Bacillus_weihenstep.fasta...
✅ Đã đọc 6 sequences. Đang tiến hành ghi file...
   + [Saved] CP000903.1.fasta (1 contigs)
   + [Saved] CP000904.1.fasta (1 contigs)
   + [Saved] CP000905.1.fasta (1 contigs)
   + [Saved] CP000906.1.fasta (1 contigs)
   + [Saved] CP000907.1.fasta (1 contigs)
   + [Saved] CP009746.1.fasta (1 contigs)

🚀 Hoàn tất! Đã tách thành 6 chủng tại: /Users/thanhnt/Desktop/Primer_tools/Non_sensulato/Bacillus_weihenstep


In [8]:
import os
from Bio import SeqIO

# --- CẤU HÌNH QC CHO B. CEREUS ---
TARGET_DIR = "/Users/thanhnt/Desktop/Primer_tools/Non_sensulato/Bacillus_weihenstep"
MIN_SIZE = 2_000_000  # 5.0 MB
MAX_SIZE = 7_000_000  # 7.0 MB (Trừ hao cho các plasmid lớn)
MAX_N_PERCENT = 3.0   # Không quá 3% Ns (đoạn đứt gãy assembly)
MAX_CONTIGS = 500     # Quá nhiều contigs thường là assembly kém chất lượng

def run_strain_qc(directory):
    print(f"🧬 Đang tiến hành QC tại: {directory}\n" + "-"*50)
    
    files = [f for f in os.listdir(directory) if f.lower().endswith(('.fasta', '.fa', '.fna'))]
    deleted_count = 0
    passed_count = 0

    for filename in files:
        file_path = os.path.join(directory, filename)
        total_len = 0
        n_count = 0
        contig_count = 0
        
        try:
            for record in SeqIO.parse(file_path, "fasta"):
                seq_str = str(record.seq).upper()
                total_len += len(seq_str)
                n_count += seq_str.count('N')
                contig_count += 1
            
            n_pct = (n_count / total_len) * 100 if total_len > 0 else 0
            
            # KIỂM TRA ĐIỀU KIỆN
            rejection_reason = []
            if total_len < MIN_SIZE: rejection_reason.append(f"Quá nhỏ ({total_len/1e6:.2f}MB)")
            if total_len > MAX_SIZE: rejection_reason.append(f"Quá lớn ({total_len/1e6:.2f}MB)")
            if n_pct > MAX_N_PERCENT: rejection_reason.append(f"N-rate cao ({n_pct:.2f}%)")
            if contig_count > MAX_CONTIGS: rejection_reason.append(f"Quá nhiều contig ({contig_count})")

            if rejection_reason:
                print(f"❌ [DELETE] {filename}: {', '.join(rejection_reason)}")
                os.remove(file_path) # Xóa file
                deleted_count += 1
            else:
                passed_count += 1
        
        except Exception as e:
            print(f"⚠️ [ERROR] Không thể đọc {filename}: {e}")
            os.remove(file_path)
            deleted_count += 1

    print("-"*50)
    print(f"✅ Hoàn tất! Giữ lại: {passed_count} | Đã loại bỏ: {deleted_count} file không đạt chuẩn.")

if __name__ == "__main__":
    run_strain_qc(TARGET_DIR)

🧬 Đang tiến hành QC tại: /Users/thanhnt/Desktop/Primer_tools/Non_sensulato/Bacillus_weihenstep
--------------------------------------------------
❌ [DELETE] CP000905.1.fasta: Quá nhỏ (0.08MB)
❌ [DELETE] CP000904.1.fasta: Quá nhỏ (0.42MB)
❌ [DELETE] CP000906.1.fasta: Quá nhỏ (0.06MB)
❌ [DELETE] CP000907.1.fasta: Quá nhỏ (0.05MB)
--------------------------------------------------
✅ Hoàn tất! Giữ lại: 2 | Đã loại bỏ: 4 file không đạt chuẩn.


In [9]:
!python insilico_pcr_advanced.py \
    -c primers.csv \
    -t /Users/thanhnt/Desktop/Primer_tools/Non_sensulato/Bacillus_mycoides \
    -o Bacillus_mycoides_Report.csv \
    -s \
    -e 4 \
    -w 28 \
    --max_len 1500

python: can't open file '/Users/thanhnt/Desktop/Primer_tools/rational_primer_design/rational_design/insilico_pcr_advanced.py': [Errno 2] No such file or directory
